# TECLA · demostración

Extrae los candidatos a término de un corpus sin instalar nada. Pulsa ▶ en las dos celdas, por orden.

> Los textos que subas se copian a esta sesión de Google Colab: no subas documentos confidenciales ni con derechos de autor. Para trabajar con tu corpus, [instala TECLA](https://github.com/isabel-mm/TECLA).

In [ ]:
#@title 1 · Instalar TECLA (unos dos minutos) { display-mode: "form" }
import subprocess, sys, os
from IPython.display import display, HTML

def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError(f"No se pudo ejecutar: {cmd}")

REPO = "/content/TECLA"
if not os.path.isdir(REPO):
    _run(f"git clone -q https://github.com/isabel-mm/TECLA.git {REPO}")
_run(f"{sys.executable} -m pip install -q -e {REPO}")
_run(f"{sys.executable} -m spacy download es_core_news_md")
_run(f"{sys.executable} -m spacy download en_core_web_lg")
if REPO not in sys.path:
    sys.path.insert(0, REPO)
display(HTML('<div style="font:600 15px sans-serif;color:#0d9488;padding:8px 0">✓ TECLA está lista. Pulsa ▶ en la celda siguiente.</div>'))

In [ ]:
#@title 2 · Abrir TECLA { display-mode: "form" }
import csv, shutil, sys
from pathlib import Path

import ipywidgets as w
import pandas as pd
from IPython.display import HTML, display

REPO = globals().get("REPO", "/content/TECLA")
if REPO not in sys.path:
    sys.path.insert(0, REPO)
TRABAJO = Path(globals().get("TRABAJO", "/content/mi_corpus"))

from pipeline.pipeline import run_tecla_pipeline
from tecla.extractor.tbx_export import export_tbx

VIOLETA, TINTA, GRIS, LAVANDA, BORDE = "#7c3aed", "#1e1b4b", "#64748b", "#f5f3ff", "#ddd6fe"
PERFILES = {"Frecuencia": {"tfidf": 0.6, "ncvalue": 0.4},
            "Equilibrado": {"tfidf": 0.5, "ncvalue": 0.5},
            "Complejidad": {"tfidf": 0.2, "ncvalue": 0.8}}
LENGUAS = {"auto": None, "español": "es", "inglés": "en"}
EJEMPLOS = {"Ejemplo en inglés": "en", "Ejemplo en español": "es"}


def extraer(origen, subidos, lengua, monolexicos, perfil):
    """Ejecuta el flujo de TECLA y devuelve (cifras, candidatos, documentos, csv, tbx)."""
    corpus = TRABAJO / "corpus"
    shutil.rmtree(TRABAJO, ignore_errors=True)
    corpus.mkdir(parents=True)
    if origen in EJEMPLOS:
        for f in sorted((Path(REPO) / "demo" / "ejemplos" / EJEMPLOS[origen]).glob("*.txt")):
            shutil.copy(f, corpus / f.name)
        lengua = "inglés" if EJEMPLOS[origen] == "en" else "español"
    else:
        if not subidos:
            raise ValueError("Sube al menos un texto (PDF o TXT) con el botón «Subir textos».")
        for nombre, contenido in subidos.items():
            (corpus / nombre).write_bytes(contenido)

    docs, detectada = [], []
    res = run_tecla_pipeline(
        corpus_path=corpus, language=LENGUAS[lengua],
        cleaning_options={"structure": True, "remove_bibliography_flag": True,
                          "remove_captions_flag": True, "remove_section_headers_flag": True},
        extraction_options={"multiword_only": not monolexicos, "max_term_length": 5,
                            "weights": PERFILES[perfil]},
        on_progress=lambda m: None,
        on_corpus_ready=lambda d, carpeta, lang: (docs.extend(d), detectada.append(lang)),
    )
    if not res:
        raise ValueError("No se ha extraído ningún candidato. Comprueba la lengua y los textos.")
    res.sort(key=lambda t: -t["terminologicity"])

    candidatos = pd.DataFrame([{
        "Candidato": t["lemma"], "Variantes": " · ".join(t.get("variants", [])) or "—",
        "Terminologicidad": round(t["terminologicity"], 4), "TF-IDF": round(t.get("tf_idf", 0), 2),
        "C-value": round(t.get("c_value", 0), 2), "NC-value": round(t.get("nc_value", 0), 2),
        "Frecuencia": t["frequency"], "Documentos": t["doc_frequency"]} for t in res])
    candidatos.index += 1

    filas, lemas, total = [], set(), 0
    for d in docs:
        tipos = {t["lemma"] for t in d["tokens"]}
        n = len(d["tokens"]); lemas |= tipos; total += n
        filas.append({"Documento": d["doc_id"], "Tokens": n, "Oraciones": len(d.get("sentences", [])),
                      "Tipos": len(tipos), "TTR": round(len(tipos) / n, 4) if n else 0})
    cifras = {"documentos": len(docs), "tokens": total, "tipos": len(lemas), "candidatos": len(res)}

    ruta_csv, ruta_tbx = TRABAJO / "candidatos_tecla.csv", TRABAJO / "candidatos_tecla.tbx"
    campos = ["term", "lemma", "variants", "patterns", "terminologicity", "tf_idf",
              "c_value", "nc_value", "frequency", "freq_rel", "doc_frequency"]
    with open(ruta_csv, "w", encoding="utf-8", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=campos); wr.writeheader()
        for t in res:
            fila = {k: t.get(k, "") for k in campos}
            fila["patterns"] = ";".join(t.get("patterns", [])); fila["variants"] = ";".join(t.get("variants", []))
            wr.writerow(fila)
    export_tbx(res, ruta_tbx, language=(detectada[0] if detectada else "en"))
    return cifras, candidatos, pd.DataFrame(filas), ruta_csv, ruta_tbx


def _miles(n):
    return f"{int(n):,}".replace(",", " ")

def _tarjetas(cifras):
    cajas = "".join(
        f'<div style="flex:1;min-width:130px;background:#fff;border:1px solid #e2e8f0;border-radius:10px;padding:12px 14px">'
        f'<div style="font:700 22px sans-serif;color:{TINTA}">{_miles(v)}</div>'
        f'<div style="font:13px sans-serif;color:{GRIS}">{k}</div></div>' for k, v in cifras.items())
    return f'<div style="display:flex;gap:12px;flex-wrap:wrap;margin:6px 0 14px">{cajas}</div>'

def _tabla(df, n=None):
    df = df if n is None else df.head(n)
    return (df.style.set_table_styles([
        {"selector": "th", "props": f"background:{LAVANDA};color:{GRIS};font:600 12px sans-serif;text-align:right;padding:8px 12px;border-bottom:1px solid {BORDE}"},
        {"selector": "td", "props": f"font:14px sans-serif;color:{TINTA};text-align:right;padding:7px 12px;border-bottom:1px solid #eef1f4"},
        {"selector": "td.col0, th.col0", "props": "text-align:left"},
    ]).bar(subset=[c for c in ["Terminologicidad"] if c in df.columns], color="#ddd6fe", vmin=0, vmax=1)
      .format(precision=4, subset=[c for c in ["Terminologicidad", "TTR"] if c in df.columns])
      .format(precision=2, subset=[c for c in ["TF-IDF", "C-value", "NC-value"] if c in df.columns]))


# ── Interfaz ──
estilo = {"description_width": "0px"}
origen = w.ToggleButtons(options=list(EJEMPLOS) + ["Mis textos"], style={"button_width": "150px"})
subida = w.FileUpload(accept=".txt,.pdf", multiple=True, description="Subir textos",
                      layout=w.Layout(display="none", width="170px"))
lengua = w.ToggleButtons(options=list(LENGUAS), style={"button_width": "92px"})
perfil = w.ToggleButtons(options=list(PERFILES), value="Equilibrado", style={"button_width": "110px"})
mono = w.Checkbox(value=True, description="Incluir candidatos monoléxicos", indent=False)
boton = w.Button(description="▶  Ejecutar", button_style="primary",
                 layout=w.Layout(width="170px", height="40px"))
boton.style.button_color = VIOLETA
estado = w.HTML()
salida = w.Output()
b_csv = w.Button(description="⬇  Descargar CSV", layout=w.Layout(width="170px", display="none"))
b_tbx = w.Button(description="⬇  Descargar TBX", layout=w.Layout(width="170px", display="none"))
_rutas = {}

def _rotulo(t):
    return w.HTML(f'<div style="font:600 11px sans-serif;letter-spacing:.08em;color:{GRIS};text-transform:uppercase;margin-top:8px">{t}</div>')

def _al_cambiar_origen(cambio):
    propio = cambio["new"] == "Mis textos"
    subida.layout.display = "inline-flex" if propio else "none"
    lengua.disabled = not propio
origen.observe(_al_cambiar_origen, names="value")
lengua.disabled = True

def _subidos():
    v = subida.value
    if isinstance(v, dict):                      # ipywidgets 7
        return {k: x["content"] for k, x in v.items()}
    return {x["name"]: bytes(x["content"]) for x in v}   # ipywidgets 8

def _al_ejecutar(_):
    boton.disabled = True
    b_csv.layout.display = b_tbx.layout.display = "none"
    estado.value = f'<span style="font:14px sans-serif;color:{GRIS}">Procesando los textos…</span>'
    salida.clear_output()
    try:
        cifras, cand, docs, ruta_csv, ruta_tbx = extraer(origen.value, _subidos(), lengua.value, mono.value, perfil.value)
        _rutas.update(csv=ruta_csv, tbx=ruta_tbx)
        with salida:
            display(HTML(_tarjetas(cifras)))
            pestañas = w.Tab()
            t1, t2 = w.Output(), w.Output()
            with t1:
                if len(cand) > 100:
                    display(HTML(f'<div style="font:13px sans-serif;color:{GRIS}">Se muestran los 100 primeros; el CSV los incluye todos.</div>'))
                display(_tabla(cand, 100))
            with t2:
                display(_tabla(docs))
            pestañas.children = [t1, t2]
            pestañas.set_title(0, "Candidatos a término"); pestañas.set_title(1, "Corpus")
            display(pestañas)
        estado.value = ""
        b_csv.layout.display = b_tbx.layout.display = "inline-flex"
    except Exception as e:
        estado.value = f'<span style="font:14px sans-serif;color:#dc2626">{e}</span>'
    finally:
        boton.disabled = False
boton.on_click(_al_ejecutar)

def _descargar(clave):
    def f(_):
        from google.colab import files
        files.download(str(_rutas[clave]))
    return f
b_csv.on_click(_descargar("csv")); b_tbx.on_click(_descargar("tbx"))

cabecera = w.HTML(
    f'<div style="display:flex;align-items:center;gap:12px;margin-bottom:4px">'
    f'<div style="width:40px;height:40px;border-radius:10px;background:{VIOLETA};color:#fff;display:grid;place-items:center;font:800 20px sans-serif">T</div>'
    f'<div><div style="font:800 20px sans-serif;color:{TINTA};letter-spacing:.04em">TECLA</div>'
    f'<div style="font:12px sans-serif;color:{GRIS}">Term Extraction and Corpus Linguistics Application</div></div></div>')
panel = w.VBox([cabecera, _rotulo("Corpus"), w.HBox([origen, subida]), _rotulo("Lengua"), lengua,
                _rotulo("Perfil de puntuación"), perfil, mono,
                w.HBox([boton, b_csv, b_tbx], layout=w.Layout(margin="10px 0 0 0")), estado, salida],
               layout=w.Layout(padding="18px", border=f"1px solid {BORDE}"))
display(panel)

**Corpus**: los ejemplos son tres textos breves sobre lingüística de corpus, redactados para esta demostración. **Perfil**: peso del *TF-IDF* frente al *NC-value* (0,6 / 0,5 / 0,2). La evaluación del corpus y la revisión de los candidatos están en la aplicación de escritorio. Licencia MIT · [github.com/isabel-mm/TECLA](https://github.com/isabel-mm/TECLA)